<a href="https://colab.research.google.com/github/samcordner/interperability-notes-monorepo/blob/main/transformer_from_scratch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
import torch.nn as nn
import math
from dataclasses import dataclass

@dataclass
class GPTConfig:
    vocab_size: int = 50257      # GPT-2 tokenizer size, matches distilGPT-2
    d_model: int = 256           # residual stream width
    n_heads: int = 4             # must divide d_model evenly
    n_layers: int = 4
    context_len: int = 128       # max sequence length
    dropout: float = 0.1
    bias: bool = True            # whether Linear/LayerNorm layers have bias terms

In [ ]:
class Embeddings(nn.Module):
    def __init__(self, config: GPTConfig):
        super().__init__()
        self.token_emb = nn.Embedding(config.vocab_size, config.d_model)
        self.pos_emb = nn.Embedding(config.context_len, config.d_model)
        self.dropout = nn.Dropout(config.dropout)

    def forward(self, idx: torch.Tensor) -> torch.Tensor:
        # idx: (batch, seq_len) of token ids
        B, T = idx.shape
        positions = torch.arange(T, device=idx.device)  # (T,)

        tok = self.token_emb(idx)          # (B, T, d_model)
        pos = self.pos_emb(positions)      # (T, d_model), broadcasts over batch

        x = tok + pos                      # (B, T, d_model) — this IS the residual stream, initialized
        return self.dropout(x)

In [ ]:
config = GPTConfig()
emb = Embeddings(config)
x = torch.randint(0, config.vocab_size, (2, 10))  # fake batch: 2 sequences, len 10
out = emb(x)
print(out.shape)  # should be torch.Size([2, 10, 256])

torch.Size([2, 10, 256])


In [ ]:
class SingleHeadAttention(nn.Module):
    def __init__(self, config: GPTConfig, head_size: int):
        super().__init__()
        self.head_size = head_size
        self.q_proj = nn.Linear(config.d_model, head_size, bias=config.bias)
        self.k_proj = nn.Linear(config.d_model, head_size, bias=config.bias)
        self.v_proj = nn.Linear(config.d_model, head_size, bias=config.bias)
        self.dropout = nn.Dropout(config.dropout)

        # causal mask: precompute once, not a learned parameter, so register as buffer
        mask = torch.tril(torch.ones(config.context_len, config.context_len))
        self.register_buffer("causal_mask", mask)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: (B, T, d_model)
        B, T, C = x.shape

        q = self.q_proj(x)   # (B, T, head_size)
        k = self.k_proj(x)   # (B, T, head_size)
        v = self.v_proj(x)   # (B, T, head_size)

        # attention scores: how much does each query token attend to each key token
        scores = q @ k.transpose(-2, -1)          # (B, T, T)
        scores = scores / math.sqrt(self.head_size)  # scale — keeps softmax gradients sane

        # apply causal mask: token i can't attend to token j > i
        scores = scores.masked_fill(self.causal_mask[:T, :T] == 0, float('-inf'))

        attn = torch.softmax(scores, dim=-1)   # (B, T, T), rows sum to 1
        attn = self.dropout(attn)

        out = attn @ v   # (B, T, head_size) — weighted sum of value vectors
        return out

In [ ]:
config = GPTConfig()
head = SingleHeadAttention(config, head_size=64)
x = torch.randn(2, 10, config.d_model)
out = head(x)
print(out.shape)  # torch.Size([2, 10, 64])

torch.Size([2, 10, 64])


In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, config: GPTConfig):
        super().__init__()
        assert config.d_model % config.n_heads == 0, "d_model must be divisible by n_heads"
        head_size = config.d_model // config.n_heads

        self.heads = nn.ModuleList([
            SingleHeadAttention(config, head_size) for _ in range(config.n_heads)
        ])
        self.out_proj = nn.Linear(config.d_model, config.d_model, bias=config.bias)
        self.dropout = nn.Dropout(config.dropout)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # run each head independently, concat along the feature dim
        out = torch.cat([h(x) for h in self.heads], dim=-1)  # (B, T, d_model)
        out = self.out_proj(out)   # mix information across heads
        return self.dropout(out)

In [ ]:
class MLP(nn.Module):
    def __init__(self, config: GPTConfig):
        super().__init__()
        self.fc1 = nn.Linear(config.d_model, 4 * config.d_model, bias=config.bias)
        self.activation = nn.GELU()
        self.fc2 = nn.Linear(4 * config.d_model, config.d_model, bias=config.bias)
        self.dropout = nn.Dropout(config.dropout)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: (B, T, d_model) — every position processed independently
        x = self.fc1(x)         # (B, T, 4*d_model) — expand
        x = self.activation(x)  # nonlinearity — this is where the "thinking" happens
        x = self.fc2(x)         # (B, T, d_model) — project back down
        return self.dropout(x)

In [ ]:
class Block(nn.Module):
    def __init__(self, config: GPTConfig):
        super().__init__()
        self.ln1 = nn.LayerNorm(config.d_model)
        self.attn = MultiHeadAttention(config)
        self.ln2 = nn.LayerNorm(config.d_model)
        self.mlp = MLP(config)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = x + self.attn(self.ln1(x))   # attention reads normed stream, writes raw residual
        x = x + self.mlp(self.ln2(x))    # MLP reads normed stream, writes raw residual
        return x

In [ ]:
config = GPTConfig()
block = Block(config)
x = torch.randn(2, 10, config.d_model)
out = block(x)
print(out.shape)  # torch.Size([2, 10, 256])

torch.Size([2, 10, 256])


In [ ]:
class GPT(nn.Module):
    def __init__(self, config: GPTConfig):
        super().__init__()
        self.config = config
        self.embeddings = Embeddings(config)
        self.blocks = nn.ModuleList([Block(config) for _ in range(config.n_layers)])
        self.ln_final = nn.LayerNorm(config.d_model)
        self.lm_head = nn.Linear(config.d_model, config.vocab_size, bias=False)

        # weight tying: share embedding and unembedding weights
        self.lm_head.weight = self.embeddings.token_emb.weight

    def forward(self, idx: torch.Tensor, targets: torch.Tensor = None):
        # idx: (B, T) token ids
        x = self.embeddings(idx)          # (B, T, d_model)

        for block in self.blocks:
            x = block(x)                   # (B, T, d_model)

        x = self.ln_final(x)               # final norm before reading out
        logits = self.lm_head(x)           # (B, T, vocab_size)

        loss = None
        if targets is not None:
            # flatten batch and time dims for cross_entropy
            loss = nn.functional.cross_entropy(
                logits.view(-1, self.config.vocab_size),
                targets.view(-1)
            )
        return logits, loss

In [ ]:
config = GPTConfig()
model = GPT(config)

x = torch.randint(0, config.vocab_size, (2, 10))
logits, loss = model(x)
print(logits.shape)  # torch.Size([2, 10, 50257])
print(loss)  # None, no targets given

targets = torch.randint(0, config.vocab_size, (2, 10))
logits, loss = model(x, targets)
print(loss)  # a scalar tensor, should be roughly log(vocab_size) ≈ 10.8 for an untrained model

torch.Size([2, 10, 50257])
None
tensor(164.5992, grad_fn=<NllLossBackward0>)
